# How `whisper` understands audio

The "ideal" input sample of audio ($\vec{x}$) that whisper expects is:
- **Sample Rate**: `16000Hz`
- **Number of channels**: `1` (mono-channel)
- **Amplitude / Data Type**: `1D float32` array
- **Duration**: `30s`

$$
\vec{x} \in \mathbb{R}^N = (x_0, \dots, x_{480000-1}) \; | \; x_i \in [-1,1] \subseteq \mathbb{R}
$$

> A 1D array with $N = \text{duration} \times \text{sample\_rate} = 480000$ samples

>[!tip] Bitrate <br>
> $\text{Bitrate} = \text{Sample Rate} \times \text{Bits per Sample} \times \text{Channels}$                                           


In [1]:
from pathlib import Path

import av
import numpy as np

AUDIO_CHUNK_PATH = Path(
    "/mnt/antonioSSD/audio-samples/samples/a_ia_ta_ai-1min/30/0-30.wav"
)
AUDIO_CHUNK_PATH.exists()

True

### `pcm_s16le`

- **pcm** (_Pulse-Code Modulation_): Uncompressed raw digital audio amplitude values.                                                        
- **s16** (_Signed 16-bit_): Each sample is a 16-bit signed integer in the range $[-32768, 32767]$ (i.e., $[-2^{15}, 2^{15}-1]$).            
- **le** (_Little-Endian_): The byte order stored in memory (least significant byte first).

In [2]:
!ffprobe -hide_banner /mnt/antonioSSD/audio-samples/samples/a_ia_ta_ai-1min/30/0-30.wav

Input #0, wav, from '/mnt/antonioSSD/audio-samples/samples/a_ia_ta_ai-1min/30/0-30.wav':
  Metadata:
    encoder         : Lavf62.12.102
  Duration: 00:00:30.00, bitrate: 256 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 16000 Hz, 1 channels, s16, 256 kb/s


In [4]:
!ffmpeg -hide_banner -formats | grep PCM

 DE  alaw            PCM A-law
 DE  f32be           PCM 32-bit floating-point big-endian
 DE  f32le           PCM 32-bit floating-point little-endian
 DE  f64be           PCM 64-bit floating-point big-endian
 DE  f64le           PCM 64-bit floating-point little-endian
 DE  mulaw           PCM mu-law
 DE  s16be           PCM signed 16-bit big-endian
 DE  s16le           PCM signed 16-bit little-endian
 DE  s24be           PCM signed 24-bit big-endian
 DE  s24le           PCM signed 24-bit little-endian
 DE  s32be           PCM signed 32-bit big-endian
 DE  s32le           PCM signed 32-bit little-endian
 DE  s8              PCM signed 8-bit
 DE  u16be           PCM unsigned 16-bit big-endian
 DE  u16le           PCM unsigned 16-bit little-endian
 DE  u24be           PCM unsigned 24-bit big-endian
 DE  u24le           PCM unsigned 24-bit little-endian
 DE  u32be           PCM unsigned 32-bit big-endian
 DE  u32le           PCM unsigned 32-bit little-endian
 DE  u8              PCM unsign

In [5]:
# frame is an av.Frame object, we must convert to `ndarray`
# `frame.to_ndarray()` returns this way: 
#   array([[-29, ..., -76]], dtype=int16)

with av.open(str(AUDIO_CHUNK_PATH)) as container:
    stream = container.streams.audio[0]
    # Extract all audio frames and concatenate them into a single numpy array
    audio_frames = []
    for frame in container.decode(stream):
        audio_frames.append(frame.to_ndarray().flatten())
    
    audio_array = np.concatenate(audio_frames)

print(f"Audio shape: {audio_array.shape}")
print(f"Data type: {audio_array.dtype}")
audio_array

Audio shape: (480000,)
Data type: int16


array([ -29,  -43,  -45, ...,  320,  725, 1117],
      shape=(480000,), dtype=int16)

### A little bit about bit structure

| Type     | Total Bits | Sign Bit | Exponent Bits | Mantissa Bits (Precision) | Max Value |
|----------|------------|----------|---------------|---------------------------|-----------|
| float32  | 32         | 1        | 8             | 23                        | $\approx 3.4 \times 10^{38}$ |
| float16  | 16         | 1        | 5             | 10                        | $\approx 6.5 \times 10^{4}$ |
| bfloat16 | 16         | 1        | 8             | 7                         | $\approx 3.4 \times 10^{38}$ |

`bfloat16` was developed by [Google](https://cloud.google.com/blog/products/ai-machine-learning/bfloat16-the-secret-to-high-performance-on-cloud-tpus) specifically for deep learning:


```python
import torch

audio_tensor_bf16 = torch.tensor(
    audio_array/np.abs(MIN_INTEGER_16),
    dtype=torch.bfloat16
)
```

In [6]:
MIN_INTEGER_16 = np.iinfo(np.int16).min

# The original dtype withot cast is: dtype('float64')
audio_array_norm = (audio_array/np.abs(MIN_INTEGER_16)).astype(np.float32)

#NOTE: we can also devide by (MAX_INTEGER_16+1).